## **Keras to HLS4ML Implementation**

## Imports

In [1]:
from tensorflow.keras.utils import to_categorical
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

%matplotlib inline
import tensorflow as tf

import os

2026-05-07 15:26:14.279117: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-07 15:26:14.281423: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-07 15:26:14.322515: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-07 15:26:14.322546: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-07 15:26:14.323719: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

Full GRU model path

In [ ]:
from tensorflow.keras.models import load_model

model = load_model('model_gru.h5')
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 layer3 (Dense)              (None, 64)                1344      
                                                                 
 relu_0 (Activation)         (None, 64)                0         
                                                                 
 output_sigmoid (Dense)      (None, 1)                 65        
                                                                 
 output_sigmoid_sigmoid (Ac  (None, 1)                 0         
 tivation)                                                       
                                                                 
Total params: 1409 (5.50 KB)
Trainable params: 1409 (5.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


# Convert Keras to HLS4ML

In [ ]:
import hls4ml
from tensorflow.keras.utils import plot_model

model = load_model('model_gru.h5')

config = hls4ml.utils.config_from_keras_model(model, granularity='name', default_precision='ap_fixed<16,6>')

hls_model = hls4ml.converters.convert_from_keras_model(
    model, hls_config=config, output_dir='GRU-Handmade/hls4ml_prj', part='xc7vx690tffg1761-2'
)


# Plot Layers

In [4]:
hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True, to_file="toptagging_gru_layers.png")
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 layer1 (GRU)                (None, 20)                1680      
                                                                 
 layer3 (Dense)              (None, 64)                1344      
                                                                 
 relu_0 (Activation)         (None, 64)                0         
                                                                 
 output_sigmoid (Dense)      (None, 1)                 65        
                                                                 
Total params: 3089 (12.07 KB)
Trainable params: 3089 (12.07 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


# Compile HLS Model

In [5]:
hls_model.compile()

In [6]:
import os

os.environ["XILINX_VIVADO"] = "/tools/Disk_Xilinx/Vivado/2019.1"
os.environ["XILINXD_LICENSE_FILE"] = "2100@xilinxd.xilinx-dev"
os.environ["PATH"] = "/tools/Disk_Xilinx/Vivado/2019.1/bin:" + os.environ.get("PATH", "")

import shutil
print("vivado_hls in notebook:", shutil.which("vivado_hls"))

vivado_hls in notebook: /tools/Disk_Xilinx/Vivado/2019.1/bin/vivado_hls


# Build HLS Model

Sanity check cell

In [ ]:
import numpy as np

x_test = np.load('x_test.npy').astype(np.float32)
y_test = np.load('y_test.npy').flatten()

hls_model.compile()

y_keras = model.predict(x_test, verbose=0).flatten()
y_hls = hls_model.predict(x_test).flatten()

acc_keras = np.mean(np.round(y_keras) == y_test)
acc_hls = np.mean(np.round(y_hls) == y_test)
print(f"Full GRU Keras accuracy : {acc_keras:.4f}")
print(f"Full GRU HLS4ML accuracy: {acc_hls:.4f}")
print(f"Mean abs diff Keras vs HLS: {np.mean(np.abs(y_keras - y_hls)):.6f}")
print(f"Max abs diff Keras vs HLS : {np.max(np.abs(y_keras - y_hls)):.6f}")


Stripped Keras accuracy : 0.5036
Stripped HLS4ML accuracy: 0.5034
Mean abs diff Keras vs HLS: 0.005031


In [8]:
hls_model.build(csim=False, synth=True, vsynth=True)


****** Vivado(TM) HLS - High-Level Synthesis from C, C++ and SystemC v2019.1 (64-bit)
  **** SW Build 2552052 on Fri May 24 14:47:09 MDT 2019
  **** IP Build 2548770 on Fri May 24 18:01:18 MDT 2019
    ** Copyright 1986-2019 Xilinx, Inc. All Rights Reserved.

source /tools/Disk_Xilinx/Vivado/2019.1/scripts/vivado_hls/hls.tcl -notrace
INFO: [HLS 200-10] Running '/tools/Disk_Xilinx/Vivado/2019.1/bin/unwrapped/lnx64.o/vivado_hls'
INFO: [HLS 200-10] For user 'leo' on host 'hvm1' (Linux_x86_64 version 5.15.0-177-generic) on Thu May 07 15:26:32 PDT 2026
INFO: [HLS 200-10] On os Ubuntu 22.04.5 LTS
INFO: [HLS 200-10] In directory '/home/leo/HLS4ML_VS_MANUAL/src/hdl/TopTagging-HLS/GRU-Handmade/hls4ml_prj_pipeclean'
Sourcing Tcl script 'build_prj.tcl'
INFO: [HLS 200-10] Opening project '/home/leo/HLS4ML_VS_MANUAL/src/hdl/TopTagging-HLS/GRU-Handmade/hls4ml_prj_pipeclean/myproject_prj'.
INFO: [HLS 200-10] Adding design file 'firmware/myproject.cpp' to the project
INFO: [HLS 200-10] Adding test be

{'CSynthesisReport': {'TargetClockPeriod': '5.00',
  'EstimatedClockPeriod': '4.236',
  'BestLatency': '13',
  'WorstLatency': '13',
  'IntervalMin': '1',
  'IntervalMax': '1',
  'BRAM_18K': '1',
  'DSP': '1092',
  'FF': '31080',
  'LUT': '39036',
  'URAM': '0',
  'AvailableBRAM_18K': '2940',
  'AvailableDSP': '3600',
  'AvailableFF': '866400',
  'AvailableLUT': '433200',
  'AvailableURAM': '0'},
 'VivadoSynthReport': {'LUT': '20745',
  'FF': '11086',
  'BRAM_18K': '0.5',
  'DSP48E': '1085'}}

# Reports

Note to self: Figure out how is Vivado working with the generated files. csim? synthesis? read_vivado_report??

In [9]:
hls4ml.report.read_vivado_report('GRU-Handmade/hls4ml_prj')

Found 1 solution(s) in GRU-Handmade/hls4ml_prj/myproject_prj.
Reports for solution "solution1":

C simulation report not found.
SYNTHESIS REPORT:
== Vivado HLS Report for 'myproject'
* Date:           Thu Apr  2 14:30:45 2026

* Version:        2019.1 (Build 2552052 on Fri May 24 15:28:33 MDT 2019)
* Project:        myproject_prj
* Solution:       solution1
* Product family: virtex7
* Target device:  xc7vx690t-ffg1761-2


== Performance Estimates
+ Timing (ns): 
    * Summary: 
    +--------+-------+----------+------------+
    |  Clock | Target| Estimated| Uncertainty|
    +--------+-------+----------+------------+
    |ap_clk  |   5.00|     4.890|        0.62|
    +--------+-------+----------+------------+

+ Latency (clock cycles): 
    * Summary: 
    +-----+-----+-----+-----+---------+
    |  Latency  |  Interval | Pipeline|
    | min | max | min | max |   Type  |
    +-----+-----+-----+-----+---------+
    |  354|  354|  354|  354|   none  |
    +-----+-----+-----+-----+---------

# Parse report

Matches extract_data in handmade sweeps

In [ ]:
import re, os

def parse_vivado_synth(filepath, features=('Slice LUTs','Slice Registers','Block RAM Tile','DSPs','Bonded IOB')):
    with open(filepath) as f:
        text = f.read()
    out = {}
    for feat in features:
        m = re.search(re.escape(feat) + r'\s*\|.*?\|.*?\|.*?\|.*?(\d+\.\d+)', text)
        out[feat] = float(m.group(1)) if m else None
    return out

rpt = 'GRU-Handmade/hls4ml_prj_pipeclean/vivado_synth.rpt'
print(rpt, '->', parse_vivado_synth(rpt) if os.path.exists(rpt) else 'NOT FOUND')

GRU-Handmade/hls4ml_prj_pipeclean/vivado_synth.rpt -> {'Slice LUTs': None, 'Slice Registers': 1.28, 'Block RAM Tile': 0.03, 'DSPs': 30.14, 'Bonded IOB': 40.47}
